# 08 Transformationskoeffizienten (Farbkamera → Johnson/Cousins)

**Wann:** nach NB02 mit RGB-Daten. Ergebnis: Koeffizienten `Tbv`, `Tv_bv` usw. in `PHOTOMETRIE_PY_DATA/transform_<Kamera>.json`, das NB03 beim Export liest.

**Vorarbeit je Kanal:** Siril CFA kalibrieren, Kanäle trennen (`tools/split_cfa.py`), je Kanal NB02 laufen lassen (`VARPHOT_TAG=<Name>_R|G|B`) mit Bildern eines Standardfelds oder des Zielfelds; Sterne unter 75 % Maximalwert, Aufnahmen bei mehreren Luftmassen.

Dann **Schritt 1** (Kamera wählen) bis **Schritt 5** (Empfehlung und Speichern) der Reihe nach ausführen. Demo-Modus läuft ohne eigene Messungen.

## Schritt 1 — Kamera, Einstellung und Eingaben

`MODE="demo"`: synthetisches Standardfeld mit **bekannten** Koeffizienten (zeigt, dass die Auswertung sie wiederfindet; es wird nichts gespeichert). `MODE="real"`: Messtabellen und Katalog laut Einstellungen.

In [ ]:
import os, datetime
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

from varphot import (PRESETS, catalog_from_vsp_table, fit_extinction, fit_transform, get_preset, instrumental_table, linfit, loo_validation,
                     read_measurements, solution_path)
from varphot.observer import camera_label, data_dir, load_config
from varphot.synthetic import standard_field
from varphot.vsp import fetch_std_field, fetch_vsp

MODE = os.environ.get("VARPHOT_NB08_MODE", "demo")                       # "demo" oder "real"
_obs, _own = load_config()
_cams = list(_obs.cameras)
if _obs.camera.model and all((c.model, c.gain_setting) != (_obs.camera.model, _obs.camera.gain_setting) for c in _cams):
    _cams.append(_obs.camera)
_opts = [(f"Profil: {camera_label(c)}", ("profil", i)) for i, c in enumerate(_cams)] + [(f"Vorlage: {p.model}", ("vorlage", k)) for k, p in PRESETS.items() if k != "eigene"]
w_cam = widgets.Dropdown(description="Kamera:", options=_opts, style={"description_width": "90px"}, layout=widgets.Layout(width="560px"))
display(w_cam)

In [ ]:
kind, ref = w_cam.value
MODEL = _cams[ref].model if kind == "profil" else get_preset(ref).model
SETTING = _cams[ref].gain_setting if kind == "profil" else next(iter(get_preset(ref).settings))
CHANNELS = {"b": "B", "g": "G", "r": "R"}                                 # Tabellenspalte → Kanalordner-Suffix
TAG = os.environ.get("VARPHOT_NB08_TAG", "standardfeld")
RESULTS = data_dir() / "results"
MEAS_FILES = {ch: RESULTS / f"{TAG}_{suf}" / "messtabelle.csv" for ch, suf in CHANNELS.items()}
STD_FIELD = os.environ.get("VARPHOT_NB08_STDFIELD", "M67")                # AAVSO-Standardfeld (per VSP-Schnittstelle): M67, NGC 7790, M11, NGC 3532, NGC 1252, Melotte 111
CATALOG_CSV = os.environ.get("VARPHOT_NB08_CATALOG")                      # oder eigene CSV mit id,ra,dec,B,V,R (hat Vorrang)
VSP_STAR = os.environ.get("VARPHOT_NB08_VSP", "Delta Cep")                # sonst: AAVSO-Sequenz dieses Sterns als Katalog
VSP_FOV, VSP_MAGLIMIT = 60, 14.0
print(f"{MODEL} — {SETTING}  (Modus {MODE})")
if MODE == "real":
    print("Messtabellen:", {k: str(v) for k, v in MEAS_FILES.items()})
    print("Katalog:", CATALOG_CSV or (f"AAVSO-Standardfeld {STD_FIELD}" if STD_FIELD else f"AAVSO-VSP von {VSP_STAR} (Feld {VSP_FOV}′, V ≤ {VSP_MAGLIMIT})"))

## Schritt 2 — Katalog und Messungen

Standardsterne mit `B`, `V` (und `R`, wenn vorhanden). Im Demo-Modus 30 synthetische Sterne mit bekannten Koeffizienten (Wahrheit: `Tb_bv` 0,25, `Tv_bv` −0,08, `Tr_bv` 0,40, Extinktion k = 0,30/0,18/0,10).

In [ ]:
if MODE == "demo":
    CAT, MEAS, TRUTH = standard_field(n_stars=30, n_frames=10, noise_mag=0.01, rng=np.random.default_rng(3))
    print("DEMO — Wahrheit:", {k: (round(v, 3) if not isinstance(v, dict) else v) for k, v in TRUTH.items()})
else:
    if CATALOG_CSV:
        CAT = pd.read_csv(CATALOG_CSV, dtype={"id": str})
    elif STD_FIELD:
        CAT = catalog_from_vsp_table(fetch_std_field(STD_FIELD), source="AAVSO-Standardfeld")      # viele Sterne, B/V/R, grosser Farbbereich
    else:
        CAT = catalog_from_vsp_table(fetch_vsp(VSP_STAR, fov_arcmin=VSP_FOV, maglimit=VSP_MAGLIMIT))   # wenige Sequenzsterne im Zielfeld
    MEAS = {ch: read_measurements(f) for ch, f in MEAS_FILES.items()}
    TRUTH = None
print(f"Katalog: {len(CAT)} Sterne, B−V {(CAT['B'] - CAT['V']).min():.2f} … {(CAT['B'] - CAT['V']).max():.2f}" + (f", mit R: {int(CAT['R'].notna().sum())}" if "R" in CAT else ""))
INSTR, FRAMES = instrumental_table(MEAS, star_ids=set(CAT["id"]))
print(f"Messungen: {INSTR['g'].notna().sum()} Sterne im Grünkanal; Bilder je Kanal: {FRAMES.groupby('channel')['file'].nunique().to_dict()}")
display(INSTR.head(5).round(3))

## Schritt 3 — Koeffizienten

Je Koeffizient eine Gerade über die Standardsterne, Ausreißer (> 3σ) werden herausgenommen. Extinktion `k` aus dem Gang des Bildnullpunkts mit der Luftmasse (nur sinnvoll bei klarer Nacht und Luftmassenbereich ≥ 0,3; bei engen Feldern wird sie nicht gebraucht).

In [ ]:
SOL = fit_transform(CAT, INSTR)
SOL.camera_model, SOL.camera_setting = MODEL, SETTING
SOL.k_extinction = fit_extinction(FRAMES)
rows = [("Tbv", SOL.Tbv, SOL.Tbv_err), ("Tb_bv", SOL.Tb_bv, SOL.Tb_bv_err), ("Tv_bv", SOL.Tv_bv, SOL.Tv_bv_err), ("Tr_bv", SOL.Tr_bv, SOL.Tr_bv_err)]
tab = pd.DataFrame([dict(Koeffizient=n, Wert=v, Fehler=e, Wahrheit=(TRUTH or {}).get(n)) for n, v, e in rows]).set_index("Koeffizient")
display(tab.round(3))
print("Extinktion k [mag/Luftmasse]:", {k: round(v, 3) for k, v in SOL.k_extinction.items()} or "nicht bestimmt (Luftmassenbereich < 0,3)", " Wahrheit:", (TRUTH or {}).get("k"))
print(f"Farbbereich der Standardsterne B−V {SOL.bv_range[0]:.2f} … {SOL.bv_range[1]:.2f} ({SOL.n_stars} Sterne) — nur darin gelten die Koeffizienten")
for w in SOL.warnings:
    print("⚠", w)
bv = (CAT.set_index("id").loc[INSTR.index.intersection(CAT["id"]), "B"] - CAT.set_index("id").loc[INSTR.index.intersection(CAT["id"]), "V"]).to_numpy()
ids = INSTR.index.intersection(CAT["id"]); C = CAT.set_index("id").loc[ids]; I = INSTR.loc[ids]
fig, ax = plt.subplots(1, 4, figsize=(15, 3.4))
panels = [("(b−g)", (I["b"] - I["g"]).to_numpy(), "Farbe: Steigung = 1/Tbv"), ("(B−b)", (C["B"] - I["b"]).to_numpy(), "Tb_bv"),
          ("(V−g)", (C["V"] - I["g"]).to_numpy(), "Tv_bv"), ("(R−r)", (C["R"] - I["r"]).to_numpy() if "R" in C else np.full(len(ids), np.nan), "Tr_bv")]
for a, (lab, y, title) in zip(ax, panels):
    ok = np.isfinite(y)
    if ok.sum() >= 4:
        f = linfit(bv[ok], y[ok]); a.plot(bv[ok][f["mask"][ok] if len(f["mask"]) == len(y) else slice(None)], y[ok][f["mask"][ok] if len(f["mask"]) == len(y) else slice(None)], "o", ms=4)
        xs = np.linspace(bv.min(), bv.max(), 10); a.plot(xs, f["slope"] * xs + f["intercept"], "r-", lw=1)
    a.set_xlabel("B−V (Katalog)"); a.set_ylabel(lab + " [mag]"); a.set_title(f"{title}", fontsize=9)
plt.tight_layout(); plt.show()

## Schritt 4 — Gütetest (Leave-one-out)

Jeder Standardstern wird als „Veränderlicher“ gegen alle anderen als Comps transformiert und mit seinem Katalogwert verglichen; zum Vergleich der **nicht transformierte** Wert (`TG` + V-Katalog). Das Handbuch nennt für geeignete Sterne
transformiertes V innerhalb ~0,02 mag (Tab. 6.1); AAVSO-Beobachter halten 0,04–0,06 mag für meldbar. Entscheidungsregel (Startwerte): ein Band wird **transformiert gemeldet**, wenn die Transformation die Abweichung verkleinert
**und** rms ≤ 0,06 mag; sonst bleibt es `TB`/`TG`/`TR` (`TRANS=NO`).

In [ ]:
VAL = loo_validation(CAT, INSTR, SOL)
SOL.validation = {b: {k: v for k, v in d.items() if not k.startswith("resid")} for b, d in VAL.items()}
rows, REPORT = [], {}
for b, d in VAL.items():
    better = d["rms_transformed"] < d["rms_standardized"]
    ok = better and d["rms_transformed"] <= 0.06
    code_trans, code_raw = {"V": ("V", "TG"), "B": ("B", "TB"), "R": ("R", "TR")}[b]
    REPORT[b] = dict(filter=code_trans if ok else code_raw, trans="YES" if ok else "NO")
    rows.append(dict(Band=b, n=d["n"], rms_nicht_transformiert=d["rms_standardized"], rms_transformiert=d["rms_transformed"], mittlere_Abweichung=d["mean"],
                     Meldung=f"{REPORT[b]['filter']}, TRANS={REPORT[b]['trans']}"))
display(pd.DataFrame(rows).set_index("Band").round(3))
fig, ax = plt.subplots(1, len(VAL), figsize=(4 * len(VAL), 3))
for a, (b, d) in zip(np.atleast_1d(ax), VAL.items()):
    a.hist(d["resid_standardized"], bins=12, alpha=0.5, label="nicht transformiert"); a.hist(d["resid_transformed"], bins=12, alpha=0.6, label="transformiert")
    a.set_title(f"{b}: Abweichung vom Katalog [mag]"); a.legend(fontsize=7)
plt.tight_layout(); plt.show()

## Schritt 5 — Empfehlung und Speichern

Bedingungen aus dem AAVSO-Forum (Hawkins/Axelsen): Ziele mit **(V−R) < 1**, **mindestens 4 Comps**; bei starker Emission/Absorption nicht transformieren (Handbuch §6.4). Die Koeffizienten gelten nur für **diesen Aufbau**
(Kamera, Einstellung, Sperrfilter, Optik) und den Farbbereich der Standardsterne. Gespeichert wird nur im Modus `real`.

In [ ]:
print(f"Meldung je Band: " + ", ".join(f"{b} → {r['filter']} (TRANS={r['trans']})" for b, r in REPORT.items()))
SOL.notes = (f"{MODE}; Katalog {CATALOG_CSV or VSP_STAR}; Meldung: " + ", ".join(f"{b}:{r['filter']}/{r['trans']}" for b, r in REPORT.items()) +
             f"; erzeugt {datetime.date.today().isoformat()}")
if MODE == "real":
    SOL.created = datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds")
    print("gespeichert:", SOL.save(solution_path(MODEL, SETTING)))
else:
    print("DEMO: nichts gespeichert.")